# 03 - Data Mining: Association Rules & Clustering
## Khai phá tri thức – Pattern Mining, Phân cụm, Phát hiện bất thường

### Nội dung:
1. **Apriori**: Tìm combo điều kiện máy liên quan lỗi → Top luật + insight vận hành
2. **Clustering**: Phân cụm máy/chu kỳ theo hành vi (KMeans, DBSCAN, HAC) → Profiling + lịch bảo trì
3. **Anomaly Detection**: So sánh anomaly vs failure thực tế

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import json
import warnings
warnings.filterwarnings('ignore')

from src.data.loader import load_params, load_raw_data
from src.features.builder import FeatureBuilder
from src.mining.association import AssociationMiner
from src.mining.clustering import ClusterAnalyzer
from src.mining.anomaly import AnomalyDetector
from src.visualization.plots import (
    plot_elbow, plot_silhouette_scores, plot_cluster_profiles, save_fig
)

params = load_params('../configs/params.yaml')
print('Config loaded ✓')

In [ ]:
# Load raw data (cần cả Type gốc cho Apriori)
df_raw = load_raw_data(path='../data/raw/ai4i2020.csv')

# Load processed data
df_processed = pd.read_parquet('../data/processed/ai4i2020_processed.parquet')

# Load feature info
with open('../data/processed/feature_info.json', 'r') as f:
    feature_info = json.load(f)

feature_cols = feature_info['feature_cols']
target_col = feature_info['target_col']
print(f'Raw: {df_raw.shape}, Processed: {df_processed.shape}')

## 1. Association Rules (Apriori) – Tìm Pattern Lỗi

**Mục tiêu:** Rời rạc hoá trạng thái máy (bin sensor/setting), chạy Apriori tìm combo điều kiện liên quan lỗi.

**Tham số:** min_support=0.01, min_confidence=0.5, min_lift=1.5

In [ ]:
# 1.1 Tạo binary DataFrame cho Apriori
builder = FeatureBuilder(params)
binary_df = builder.get_apriori_features(df_raw, params)
print(f'Binary DataFrame shape: {binary_df.shape}')
print(f'Columns: {binary_df.columns.tolist()}')
binary_df.head()

In [ ]:
# 1.2 Chạy Apriori
miner = AssociationMiner(params)
freq_itemsets, rules = miner.mine(binary_df)

print(f'\nFrequent itemsets: {len(freq_itemsets)}')
print(f'Association rules: {len(rules)}')

if len(freq_itemsets) > 0:
    print('\nTop 10 frequent itemsets by support:')
    display(freq_itemsets.nlargest(10, 'support'))

In [ ]:
# 1.3 Top luật kết hợp
if len(rules) > 0:
    print('=== TOP 20 LUẬT KẾT HỢP (theo Lift) ===')
    top_rules = miner.get_top_rules(n=20, sort_by='lift')
    display(top_rules[['antecedents', 'consequents', 'support', 'confidence', 'lift']])
else:
    print('Không tìm thấy luật. Thử giảm min_support hoặc min_confidence.')

In [ ]:
# 1.4 Luật liên quan Machine failure
failure_rules = miner.get_failure_rules('Machine failure')
if len(failure_rules) > 0:
    print('=== LUẬT CÓ CONSEQUENT = Machine failure ===')
    display(failure_rules[['antecedents', 'consequents', 'support', 'confidence', 'lift']].head(15))
    
    # Luật dạng text
    print('\n--- Dạng text ---')
    for text in miner.rules_to_text(failure_rules, top_n=10):
        print(text)
else:
    print('Không tìm thấy luật liên quan Machine failure.')

In [ ]:
# 1.5 Luật theo từng loại lỗi
failure_types = params['data']['failure_types']
ft_rules = miner.get_failure_type_rules(failure_types)

for ft, ft_df in ft_rules.items():
    print(f'\n=== {ft} Rules ===')
    display(ft_df[['antecedents', 'consequents', 'support', 'confidence', 'lift']].head(5))

In [ ]:
# 1.6 Visualize top rules
if len(rules) > 0:
    fig, axes = plt.subplots(1, 2, figsize=(14, 6))
    
    # Support vs Confidence
    axes[0].scatter(rules['support'], rules['confidence'], 
                    c=rules['lift'], cmap='RdYlGn', alpha=0.6, s=50)
    axes[0].set_xlabel('Support')
    axes[0].set_ylabel('Confidence')
    axes[0].set_title('Support vs Confidence (color=Lift)')
    plt.colorbar(axes[0].collections[0], ax=axes[0], label='Lift')
    
    # Lift distribution
    axes[1].hist(rules['lift'], bins=30, color='#e74c3c', edgecolor='white')
    axes[1].set_xlabel('Lift')
    axes[1].set_ylabel('Count')
    axes[1].set_title('Lift Distribution')
    
    plt.tight_layout()
    save_fig(fig, '03_association_rules', '../outputs/figures')
    plt.show()

### Insight từ Association Rules
- Các combo điều kiện (nhiệt độ cao + tốc độ thấp + torque cao) có lift cao → liên quan lỗi HDF, OSF
- Tool wear cao kết hợp torque cao → Overstrain Failure
- Power ngoài ngưỡng → Power Failure

---
## 2. Clustering – Phân cụm máy/chu kỳ theo hành vi

**Mục tiêu:** Phân cụm các quan sát dựa trên đặc trưng sensor → Profiling cụm → Đề xuất lịch bảo trì

**Thuật toán:** KMeans, DBSCAN, Hierarchical (HAC)

In [ ]:
# 2.1 Chuẩn bị dữ liệu cho clustering (dùng sensor features, đã scale)
from sklearn.preprocessing import StandardScaler

# Dùng chỉ các sensor features gốc (đã clean)
cluster_features = ['Air temperature [K]', 'Process temperature [K]', 
                     'Rotational speed [rpm]', 'Torque [Nm]', 'Tool wear [min]']
cluster_features = [c for c in cluster_features if c in df_processed.columns]

X_cluster = df_processed[cluster_features].values
scaler = StandardScaler()
X_cluster_scaled = scaler.fit_transform(X_cluster)

print(f'Clustering data shape: {X_cluster_scaled.shape}')
print(f'Features: {cluster_features}')

In [ ]:
# 2.2 KMeans – Elbow Method & Silhouette
cluster_analyzer = ClusterAnalyzer(params)
kmeans_results = cluster_analyzer.fit_kmeans(X_cluster_scaled)

# Elbow plot
inertias = {k: v['inertia'] for k, v in kmeans_results.items()}
fig = plot_elbow(inertias)
save_fig(fig, '03_elbow_method', '../outputs/figures')
plt.show()

# Silhouette plot
sil_scores = {k: v['silhouette'] for k, v in kmeans_results.items()}
fig = plot_silhouette_scores(sil_scores)
save_fig(fig, '03_silhouette_scores', '../outputs/figures')
plt.show()

In [ ]:
# 2.3 DBSCAN
dbscan_results = cluster_analyzer.fit_dbscan(X_cluster_scaled)

# Show DBSCAN results
print('\nDBSCAN Results:')
for key, val in dbscan_results.items():
    print(f"  {key}: clusters={val['n_clusters']}, noise={val['n_noise']}, sil={val['silhouette']:.4f}")

In [ ]:
# 2.4 Hierarchical Clustering (HAC)
hac_results = cluster_analyzer.fit_hierarchical(X_cluster_scaled)

print('\nHAC Results:')
for k, val in hac_results.items():
    print(f"  k={k}: sil={val['silhouette']:.4f}, dbi={val['davies_bouldin']:.4f}")

In [ ]:
# 2.5 So sánh tất cả clustering models
scores_table = cluster_analyzer.get_scores_table()
print('=== CLUSTERING COMPARISON ===')
display(scores_table)

# Lưu bảng
scores_table.to_csv('../outputs/tables/clustering_comparison.csv', index=False)

# Best model
best_name, best_labels = cluster_analyzer.get_best_model(metric='silhouette')

In [ ]:
# 2.6 Profiling cụm tốt nhất
profile = cluster_analyzer.profile_clusters(df_processed, best_labels, cluster_features)

# Cluster profile visualization
cluster_means = df_processed[cluster_features].copy()
cluster_means['cluster'] = best_labels
cluster_means = cluster_means[cluster_means['cluster'] != -1].groupby('cluster').mean()

# Normalize for comparison
cluster_means_norm = (cluster_means - cluster_means.min()) / (cluster_means.max() - cluster_means.min() + 1e-10)

fig = plot_cluster_profiles(cluster_means_norm, cluster_features)
save_fig(fig, '03_cluster_profiles', '../outputs/figures')
plt.show()

In [ ]:
# 2.7 Failure rate per cluster (→ lịch bảo trì)
df_cluster = df_processed.copy()
df_cluster['cluster'] = best_labels
df_cluster_valid = df_cluster[df_cluster['cluster'] != -1]

failure_by_cluster = df_cluster_valid.groupby('cluster').agg(
    count=('Machine failure', 'count'),
    n_failures=('Machine failure', 'sum'),
    failure_rate=('Machine failure', 'mean'),
    avg_tool_wear=('Tool wear [min]', 'mean'),
    avg_torque=('Torque [Nm]', 'mean'),
    avg_speed=('Rotational speed [rpm]', 'mean'),
).round(4)

print('=== CLUSTER FAILURE PROFILES ===')
display(failure_by_cluster)

# Visualize
fig, ax = plt.subplots(figsize=(10, 5))
ax.bar(failure_by_cluster.index.astype(str), failure_by_cluster['failure_rate'] * 100, 
       color=plt.cm.RdYlGn_r(failure_by_cluster['failure_rate'] / failure_by_cluster['failure_rate'].max()))
ax.set_xlabel('Cluster')
ax.set_ylabel('Failure Rate (%)')
ax.set_title('Failure Rate by Cluster → Maintenance Priority')
for i, (idx, row) in enumerate(failure_by_cluster.iterrows()):
    ax.text(i, row['failure_rate']*100 + 0.3, f"{row['failure_rate']*100:.1f}%", ha='center', fontweight='bold')
plt.tight_layout()
save_fig(fig, '03_failure_by_cluster', '../outputs/figures')
plt.show()

# Save
failure_by_cluster.to_csv('../outputs/tables/cluster_failure_profiles.csv')

In [ ]:
# 2.8 2D Visualization (PCA) of clusters
from sklearn.decomposition import PCA

pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_cluster_scaled)

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Clusters
scatter = axes[0].scatter(X_pca[:, 0], X_pca[:, 1], c=best_labels, cmap='tab10', alpha=0.4, s=5)
axes[0].set_title(f'Clusters ({best_name})')
axes[0].set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]*100:.1f}%)')
axes[0].set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]*100:.1f}%)')
plt.colorbar(scatter, ax=axes[0])

# Actual failures
colors = df_processed['Machine failure'].map({0: '#2ecc71', 1: '#e74c3c'})
axes[1].scatter(X_pca[:, 0], X_pca[:, 1], c=colors, alpha=0.4, s=5)
axes[1].set_title('Actual Failures (red=failure)')
axes[1].set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]*100:.1f}%)')
axes[1].set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]*100:.1f}%)')

plt.tight_layout()
save_fig(fig, '03_pca_clusters_vs_failures', '../outputs/figures')
plt.show()

---
## 3. Anomaly Detection

In [ ]:
# 3.1 Chạy anomaly detection
detector = AnomalyDetector(params)

# Contamination ≈ actual failure rate
contamination = df_processed['Machine failure'].mean()
print(f'Contamination (actual failure rate): {contamination:.4f}')

detector.fit_isolation_forest(X_cluster_scaled, contamination=contamination)
detector.fit_lof(X_cluster_scaled, contamination=contamination)
# One-Class SVM trên subsample (chậm)
idx_sample = np.random.RandomState(42).choice(len(X_cluster_scaled), 3000, replace=False)
# detector.fit_ocsvm(X_cluster_scaled[idx_sample], nu=contamination)

In [ ]:
# 3.2 So sánh anomaly detection vs actual failures
y_true = df_processed['Machine failure'].values
anomaly_comparison = detector.compare_with_actual(y_true)
display(anomaly_comparison)

# Save
anomaly_comparison.to_csv('../outputs/tables/anomaly_comparison.csv', index=False)

---
## 4. Tóm tắt Mining

### Kết quả Apriori:
- Tìm được các combo điều kiện liên quan lỗi (lift cao)
- Nhóm điều kiện rủi ro: temp_diff thấp + speed thấp → HDF; tool wear cao + torque cao → OSF

### Kết quả Clustering:
- Phân cụm cho thấy nhóm hành vi máy khác nhau về failure rate
- Cụm có failure rate cao → ưu tiên bảo trì

### Kết quả Anomaly Detection:
- Isolation Forest/LOF phát hiện anomaly tương quan với failure thực tế
- Recall/Precision phản ánh khó khăn khi failure rate thấp